# HW3 — Networks, Rankings, and Academic Prestige
**Applied Machine Learning for Business**  
**Spring 2026 | Plaksha University**  
**Faculty Hiring as a Prestige Network**  
**Instructor: Nikhil George**

## Overview

When a university hires a new faculty member, it leaves a trace: the hiring institution, the PhD-granting institution, and the direction of prestige flow. Aggregated across thousands of hires, these traces form a network. This assignment asks whether that network encodes academic prestige — and whether graph algorithms can recover rankings that match external judgments like US News.

You will implement three algorithms from scratch: eigenvector centrality via power iteration, a random walk, and PageRank. You will run them on two hiring networks (CS and Business), compare the results to US News rankings, and confront one case where the algorithm produces a counterintuitive answer.

**Submission:** Individual or pairs. If submitting as a pair, include both names in the notebook header. Submit one notebook per group.

**Datasets**

- `cs_hiring_edges.csv` — CS faculty hiring (source = PhD-granting institution, target = hiring institution, weight = number of hires)
- `cs_hiring_nodes.csv` — CS institutions (institution, prestige_score, usnews_rank, region)
- `biz_hiring_edges.csv` — Business faculty hiring (same format)
- `biz_hiring_nodes.csv` — Business institutions (same format)

Source: Clauset, Arbesman, Larremore (2015), *Science Advances*. US News 2010 rankings for CS; US News 2012 rankings for Business — both included in the published dataset.

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

SEED = 42

In [ ]:
cs_edges  = pd.read_csv('cs_hiring_edges.csv')
cs_nodes  = pd.read_csv('cs_hiring_nodes.csv').set_index('institution')
biz_edges = pd.read_csv('biz_hiring_edges.csv')
biz_nodes = pd.read_csv('biz_hiring_nodes.csv').set_index('institution')

# Undirected graphs — used in Steps 1–3
G_cs  = nx.from_pandas_edgelist(cs_edges,  'source', 'target')
G_biz = nx.from_pandas_edgelist(biz_edges, 'source', 'target')

# Directed graphs — used in Step 4
# Edge direction: source = PhD-granting institution, target = hiring institution
DG_cs  = nx.from_pandas_edgelist(cs_edges,  'source', 'target', create_using=nx.DiGraph())
DG_biz = nx.from_pandas_edgelist(biz_edges, 'source', 'target', create_using=nx.DiGraph())

---
## Step 1: Network Basics (8 pts)

In [ ]:
for label, G in [('CS', G_cs), ('Business', G_biz)]:
    n   = G.number_of_nodes()
    e   = G.number_of_edges()
    avg = 2 * e / n
    top = max(dict(G.degree()), key=lambda v: G.degree(v))
    print(f"{label}: {n} nodes, {e} edges, avg degree = {avg:.1f}")
    print(f"  Highest-degree node: {top} (degree {G.degree(top)})\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (label, G) in zip(axes, [('CS', G_cs), ('Business', G_biz)]):
    degrees = [d for _, d in G.degree()]
    ax.hist(degrees, bins=25, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_xlabel('Degree')
    ax.set_ylabel('Count')
    ax.set_title(f'{label} Degree Distribution')
plt.tight_layout()
plt.show()

**Q1.1 (2 pts):** Report nodes, edges, and average degree for each network.

**Q1.2 (2 pts):** What does a node represent? What does an edge represent? What does the direction of an edge encode?

**Q1.3 (2 pts):** Describe the shape of the CS degree distribution. What does this shape imply about the structure of hiring relationships across departments?

**Q1.4 (2 pts):** The highest-degree node in the CS network has connections to more departments than any other. Is this consistent with US News prestige? What does high degree mean in this network?

**Answers — Step 1:**

*Q1.1:*

*Q1.2:*

*Q1.3:*

*Q1.4:*

---
## Step 2: Eigenvector Centrality via Power Iteration (18 pts)

We treat the hiring network as undirected here — the same institutions, but asking a mutual-embeddedness question rather than a flow question. EC is the right tool for that question; Step 4 introduces direction.

Eigenvector centrality (EC) assigns each node a score equal to the weighted sum of its neighbors' scores. The scores satisfy:

$$\mathbf{v} = \frac{1}{\lambda} A \mathbf{v}$$

where $A$ is the adjacency matrix and $\lambda$ is the largest eigenvalue. Power iteration computes this iteratively: multiply by $A$, normalize, repeat until convergence.

In [ ]:
def power_iteration(G, max_iter=300, tol=1e-10):
    """Eigenvector centrality via power iteration (unweighted)."""
    nodes = list(G.nodes())
    n     = len(nodes)
    A     = nx.to_numpy_array(G, nodelist=nodes, weight=None)  # unweighted adjacency

    v = np.ones(n) / np.sqrt(n)  # uniform initialization

    for i in range(max_iter):
        # YOUR CODE: compute the next iterate
        # Hint: v_new = matrix-vector product of A and v
        v_new = ___

        v_new = v_new / np.linalg.norm(v_new)   # normalize to unit length

        if np.linalg.norm(v_new - v) < tol:
            print(f"Converged in {i + 1} iterations.")
            break
        v = v_new

    return pd.Series(v_new, index=nodes, name='ec_score')

In [ ]:
ec_cs = power_iteration(G_cs)

cs_ec = cs_nodes[['usnews_rank']].copy()
cs_ec['ec_score'] = ec_cs
cs_ec = cs_ec.dropna(subset=['usnews_rank'])
cs_ec['ec_rank'] = cs_ec['ec_score'].rank(ascending=False).astype(int)

print("Top 10 CS departments by eigenvector centrality:\n")
top10 = cs_ec.sort_values('ec_score', ascending=False).head(10)
print(top10[['ec_score', 'ec_rank', 'usnews_rank']].round(4).to_string())

**Q2.1 (4 pts):** Report the top 10 CS departments by EC score, alongside their EC rank and US News rank.

> **Before reading further:** Will EC rank agree with US News rank for CS departments? In what kind of department might EC rank substantially higher than US News would suggest — and why?

In [ ]:
# Q2.2: YOUR CODE — compute Spearman r between ec_rank and usnews_rank
# Use cs_ec['ec_rank'] and cs_ec['usnews_rank']
corr_cs, pval_cs = ___
print(f"CS: Spearman r = {corr_cs:.3f},  p = {pval_cs:.4f}")

**Q2.2 (3 pts):** Report Spearman $r$ and interpret — does EC recover the US News ranking?

**Q2.3 (3 pts):** Identify one CS department whose EC rank and US News rank differ by more than 10 positions. What structural property of the hiring network might explain the gap?

**Answers — Q2.1–2.3:**

*Q2.1:* *(table printed above — reference it here)*

*Q2.2:*

*Q2.3:*

**Q2.4–2.5:** Repeat the EC analysis for the Business network.

In [ ]:
# Q2.4: Run EC on Business network
ec_biz = power_iteration(G_biz)

biz_ec = biz_nodes[['usnews_rank']].copy()
biz_ec['ec_score'] = ec_biz
biz_ec = biz_ec.dropna(subset=['usnews_rank'])
biz_ec['ec_rank'] = biz_ec['ec_score'].rank(ascending=False).astype(int)

# YOUR CODE: print top 10 by EC and compute Spearman for Business
print("Top 10 Business departments by eigenvector centrality:\n")
___

corr_biz, pval_biz = ___
print(f"\nBusiness: Spearman r = {corr_biz:.3f},  p = {pval_biz:.4f}")

**Q2.4 (4 pts):** Report the top 10 Business departments by EC score, with EC rank and US News rank.

**Q2.5 (4 pts):** Is the Spearman $r$ for Business higher or lower than CS? Propose one structural reason why the two networks might differ in how well EC recovers US News prestige.

**Answers — Q2.4–2.5:**

*Q2.4:* *(table printed above)*

*Q2.5:*

---
## Step 3: Random Walk and the Stationary Distribution (12 pts)

A random walker on a graph starts at some node and at each step moves uniformly at random to a neighbor. After enough steps, the fraction of time spent at each node converges to the **stationary distribution** of the walk.

> **Before running Q3.1:** A random walk on an undirected graph is guaranteed to converge. The stationary probability of visiting node $i$ depends only on a local property of that node — not on the global structure of the network. What property is that? Make a prediction before you run the simulation.

In [ ]:
def random_walk(G, n_steps=100_000, seed=SEED):
    """Simulate a random walk on an undirected graph.
    Returns a Series: {node: visit_frequency}, summing to ~1."""
    rng     = np.random.default_rng(seed)
    nodes   = list(G.nodes())
    current = rng.choice(nodes)         # start at a random node
    visits  = {node: 0 for node in nodes}

    for _ in range(n_steps):
        neighbors = list(G.neighbors(current))
        # YOUR CODE: move to a uniformly random neighbor
        current = ___
        visits[current] += 1

    total = sum(visits.values())
    return pd.Series({k: v / total for k, v in visits.items()}, name='visit_freq')

In [ ]:
visit_cs = random_walk(G_cs)

degree_cs = pd.Series(dict(G_cs.degree()), name='degree')
rw_cs = pd.DataFrame({'visit_freq': visit_cs, 'degree': degree_cs})
rw_cs['visit_rank']  = rw_cs['visit_freq'].rank(ascending=False).astype(int)
rw_cs['degree_rank'] = rw_cs['degree'].rank(ascending=False).astype(int)

print("Top 10 by visit frequency:\n")
print(rw_cs.sort_values('visit_freq', ascending=False).head(10).to_string())

print("\nMIT — visit rank, degree rank, US News rank:")
if 'MIT' in rw_cs.index:
    print(rw_cs.loc['MIT'])
    print(f"  US News rank: {cs_nodes.loc['MIT', 'usnews_rank']}")

**Q3.1 (3 pts):** Report the top 5 CS departments by visit frequency and their degree. What is the relationship between visit rank and degree rank?

**Q3.2 (2 pts):** MIT has US News rank 1. What is its visit rank? Is a high visit rank the same as being prestigious in this network?

**Q3.3 (3 pts):** Explain why the stationary distribution of a simple random walk on an undirected graph equals degree / (2 × edges), not a prestige-based ranking. What property of the walk determines visit frequency?

**Q3.4 (4 pts):** PageRank modifies the random walk to make it suitable for ranking. Name two specific changes PageRank makes and explain what problem each solves.

**Answers — Step 3:**

*Q3.1:*

*Q3.2:*

*Q3.3:*

*Q3.4:*

---
## Step 4: Directed Graph and PageRank (16 pts)

PageRank operates on a directed graph. A node's rank is the weighted sum of the ranks of nodes that link **to** it, divided by those nodes' out-degrees. A damping factor $d$ (typically 0.85) models the probability of following an edge rather than teleporting to a random node:

$$\mathbf{r} = d \cdot M \mathbf{r} + (1 - d) \cdot \frac{\mathbf{1}}{n}$$

where $M$ is the column-stochastic transition matrix built from the directed adjacency matrix.

> **Before running Q4.1:** In the directed graph, an edge A → B means institution A produced a PhD hired by institution B. Think about what PageRank measures given this direction: which nodes receive the most incoming rank? Predict whether MIT will be ranked #1.

In [ ]:
def pagerank(G, d=0.85, max_iter=300, tol=1e-10):
    """PageRank on a directed graph via power iteration."""
    nodes = list(G.nodes())
    n     = len(nodes)
    A     = nx.to_numpy_array(G, nodelist=nodes, weight=None)  # unweighted

    # Row-stochastic: normalize each row by out-degree
    out_deg = A.sum(axis=1, keepdims=True)
    out_deg[out_deg == 0] = 1          # handle sink nodes (no outgoing edges)
    A_norm  = A / out_deg              # row-stochastic
    M       = A_norm.T                 # column-stochastic: rank flows into columns

    rank     = np.ones(n) / n          # uniform initialization
    teleport = np.ones(n) / n

    for i in range(max_iter):
        # YOUR CODE: PageRank update rule
        # rank_new = damping * (column-stochastic matrix @ rank) + (1 - damping) * teleport
        rank_new = ___

        if np.linalg.norm(rank_new - rank) < tol:
            print(f"Converged in {i + 1} iterations.")
            break
        rank = rank_new

    return pd.Series(rank_new, index=nodes, name='pagerank')

In [ ]:
# PageRank on original direction: phd_source -> hiring_institution
pr_orig = pagerank(DG_cs)

cs_pr_orig = cs_nodes[['usnews_rank']].copy()
cs_pr_orig['pagerank'] = pr_orig
cs_pr_orig = cs_pr_orig.dropna(subset=['usnews_rank'])
cs_pr_orig['pr_rank'] = cs_pr_orig['pagerank'].rank(ascending=False).astype(int)

print("Top 10 CS by PageRank — original direction (phd_source -> hiring):\n")
print(cs_pr_orig.sort_values('pagerank', ascending=False)
      .head(10)[['pagerank', 'pr_rank', 'usnews_rank']].round(5).to_string())

**Q4.1 (3 pts):** What is the #1 CS department by PageRank in the original direction? Is this what you predicted?

**Q4.2 (4 pts):** In the original edge direction (phd_source → hiring), PageRank measures a property of each node based on who links to it. What property does it measure? Why might a department with a non-elite US News rank end up ranked #1 by this measure?

**Answers — Q4.1–4.2:**

*Q4.1:*

*Q4.2:*

> **Before running:** Flipping every edge changes what PageRank measures. The original direction (phd_source → hiring) asked one question; the reversed direction (hiring → phd_source) asks another. Which direction do you expect to rank MIT #1 — and why?

> **Reversing a directed graph:** NetworkX `DiGraph` objects have a `.reverse()` method that returns a new graph with all edge directions flipped — every A → B becomes B → A. The topology is unchanged; only the direction changes. `DG.reverse()` does not modify the original graph.

In [ ]:
# Q4.3 — YOUR CODE: reverse all edges in DG_cs
# Each original edge (phd_source -> hiring) becomes (hiring -> phd_source)
DG_cs_rev = ___

pr_rev = pagerank(DG_cs_rev)

cs_pr_rev = cs_nodes[['usnews_rank']].copy()
cs_pr_rev['pagerank'] = pr_rev
cs_pr_rev = cs_pr_rev.dropna(subset=['usnews_rank'])
cs_pr_rev['pr_rank'] = cs_pr_rev['pagerank'].rank(ascending=False).astype(int)

print("Top 10 CS by PageRank — reversed direction (hiring -> phd_source):\n")
print(cs_pr_rev.sort_values('pagerank', ascending=False)
      .head(10)[['pagerank', 'pr_rank', 'usnews_rank']].round(5).to_string())

In [ ]:
# Q4.3 — YOUR CODE: compute Spearman for reversed PageRank
# Use cs_pr_rev['pr_rank'] and cs_pr_rev['usnews_rank']
corr_pr_rev, pval_pr_rev = ___
print(f"PageRank (reversed): Spearman r = {corr_pr_rev:.3f},  p = {pval_pr_rev:.4f}")

# Compare to EC Spearman from Step 2
print(f"EC (undirected):      Spearman r = {corr_cs:.3f}")

**Q4.3 (5 pts):** Report the top 5 CS departments by reversed PageRank and the Spearman $r$ with US News. How does the ranking change compared to the original direction?

**Q4.4 (4 pts):** EC (undirected) and PageRank (reversed directed) both produce high Spearman correlations with US News. Which is higher? What does this tell you about whether directionality carries additional information about prestige in the hiring network?

**Answers — Q4.3–4.4:**

*Q4.3:*

*Q4.4:*

---
## Step 5: Synthesis (16 pts)

**Q5.1 (8 pts): The Temperature of Innovation**

An AI coding agent generates solutions by traversing a state-space graph of architectural approaches. At each node, transition probabilities are determined by:

$$P(j \mid i) = \frac{e^{z_{ij}/T}}{\sum_k e^{z_{ik}/T}}$$

The agent was trained on 50 million GitHub commit histories. Of all commits that touch sorting code, 97% use nested loops or standard library calls. Vectorized approaches appear in under 1% of training examples.

| From | To | Logit $z$ |
|---|---|---|
| S | A (standard) | 4.0 |
| S | B (pivot) | 1.0 |
| A | A (stay stuck) | 4.0 |
| A | C (global optimum) | 0.0 |
| B | B (stay broken) | 0.0 |
| B | C (global optimum) | 4.0 |

**(a)** Logit scores are raw unnormalized preferences learned from training data. What does $z_{SA} = 4.0$ versus $z_{SB} = 1.0$ tell you about this agent — and what does it not tell you?

**(b)** At $T = 1.0$, compute $P(S \to A \to C)$ and $P(S \to B \to C)$ as 2-step path probabilities: apply the softmax at S to get the first transition, then apply it again at the intermediate node. Which path gives higher probability — and where is the bottleneck on that path? Now consider: the self-loops mean the agent may revisit A or B many times before reaching C. Given unlimited steps, what is the probability of eventually reaching C from S — and what does this tell you about what the 2-step path probability is actually measuring?

**(c)** An engineering manager raises temperature to $T = 5.0$. Total $P(C)$ jumps from 6% to 45% — a 7x improvement. Yet the agent is now less certain about $B \to C$, a step it was executing correctly 98% of the time at $T = 1.0$. Explain the paradox. Give one business setting where you would run high temperature and one where you would enforce low temperature.

**Q5.2 (8 pts): Building a Recommendation Engine**

An e-commerce company wants to build a "Next Product" recommendation engine. The current site has no recommendation system — users navigate freely without algorithmic nudges. The team proposes building a random walk on a product graph: nodes are products, edges are built from user behavior data.

Before writing any code, the team must decide what an edge means.

**(a)** A colleague proposes using page visits — if a user viewed Product A then Product B, add an edge A → B. Give two reasons this produces a poor graph for recommendations.

**(b)** Another colleague proposes using purchases instead. This gives cleaner signal but creates a different structural problem. What is it?

**(c)** In plain terms, how does the random walk generate a recommendation once the graph is built?

**(d)** Your company launches a new high-margin product today. Regardless of whether you use visits or purchases to define edges, what is the probability the engine recommends it — and what is the graph-theoretic reason, not a data reason?

**(e)** The team adds a teleportation parameter: at each step, with some small probability the walker jumps uniformly to any product regardless of edges. How does this fix the problem in (d), and what is the direct connection to the PageRank you implemented in Step 4?

**(f)** The analysis above assumes no recommendation system is currently in place — users navigated without algorithmic nudges. Your company now deploys the engine. How does this change the data generated going forward? What happens to the graph structure over time — and what does this mean for products the engine initially ranks low?

**Answers — Step 5:**

*Q5.1a:*

*Q5.1b:*

*Q5.1c:*

*Q5.2a:*

*Q5.2b:*

*Q5.2c:*

*Q5.2d:*

*Q5.2e:*

*Q5.2f:*